# 04 — Scaled MAPPO Training  (Fix 3 — Reward Economy)

Multi-Agent Reinforcement Learning for crop disease monitoring (scaled):
- **Grid:** 10x10 = 100 sectors
- **UAVs:** 4 (start at four corners / Access Points)
- **Episode:** 72 days x 40 steps/day = 2,880 inner steps
- **Algorithm:** MAPPO with Transformer-based actors (~334K params each)

### Fixes applied in this notebook (Fix 3)
| # | Fix | Detail |
|---|-----|--------|
| 5 | SAFE_RETURN_BONUS 10→1 | Eliminates lazy-agent collapse; return is lack of punishment, not a goal |
| 6 | Deterministic evaluation | Uses `dist.loc` (mean action) instead of stochastic `get_action()` sample |
| 7 | Best-diagnosis checkpoint | Saves separate best weights whenever `n_diagnosed` improves |
| 8 | `ppo_update` return values | Verified returns `np.mean(actor_losses), np.mean(critic_losses)` |
| 9 | Import `uav_env_3` / `networks_3` | Wires in double-crash-penalty fix, W_UNKNOWN_FLOOR=0.1, proximity tie-breaking, PBRS_SCALE=2.0 |

### Retained fixes from Fix 2
| # | Change | Detail |
|---|--------|--------|
| 1 | Daily sortie loop | 40 inner steps/day, energy recharges, UAVs return to APs |
| 2 | PBRS reward | Explore mode → approach best unknown; Return mode → approach AP |
| 3 | Crash mechanic | Energy depletion away from AP = CRASH_PENALTY (training loop only, no double-count) |
| 4 | Split discovery bonus | INFECTED_FOUND_BONUS=30, HEALTHY_FOUND_BONUS=5 |
| 5 | No HOVER_BONUS | Set to 0.0 — eliminates corner exploitation |
| 6 | ENTROPY_COEFF=0.001 | Fixes negative actor loss in continuous MAPPO |
| 7 | Observation expansion | 211→215: AP distance, survival ratio, daily step norm |
| 8 | Larger mini-batches | 24→128 (rollout is 2,880 steps now) |
| 9 | Fewer PPO epochs | 6→4 (larger batches need fewer passes) |

**Kaggle dataset must contain these 6 files:**
```
grid_config.json     (from grid_scaled/)
sector_status.csv    (from grid_scaled/)
simulation_log.csv   (from simulation_scaled/)
dataset.npy          (from simulation_scaled/)
uav_env_3.py         (from project/fixes-1/)
networks_3.py        (from project/fixes-1/)
```
Set `DATASET_NAME` in the next cell to match your Kaggle dataset slug.

In [ ]:
import os, sys

ON_KAGGLE = os.path.exists('/kaggle/input')

if ON_KAGGLE:
    DATASET_NAME  = 'mappo-dataset-4'
    DATA_DIR      = f'/kaggle/input/datasets/prabhavsunia/{DATASET_NAME}'
    SRC_DIR       = DATA_DIR
    SIM_LOG_PATH  = os.path.join(DATA_DIR, 'simulation_log.csv')
    GRID_CFG_PATH = os.path.join(DATA_DIR, 'grid_config.json')
    DATASET_PATH  = os.path.join(DATA_DIR, 'dataset.npy')
    WEIGHTS_DIR   = '/kaggle/working'
    RESULTS_DIR   = '/kaggle/working'
else:
    _NB_DIR       = os.getcwd()
    _PROJ_DIR     = os.path.dirname(_NB_DIR)
    _ROOT_DIR     = os.path.dirname(_PROJ_DIR)
    SRC_DIR       = os.path.join(_PROJ_DIR, 'src_scaled')
    SIM_LOG_PATH  = os.path.join(_ROOT_DIR, 'simulation_scaled', 'simulation_log.csv')
    GRID_CFG_PATH = os.path.join(_ROOT_DIR, 'grid_scaled', 'grid_config.json')
    DATASET_PATH  = os.path.join(_ROOT_DIR, 'simulation_scaled', 'dataset.npy')
    WEIGHTS_DIR   = os.path.join(_PROJ_DIR, 'weights_scaled')
    RESULTS_DIR   = os.path.join(_PROJ_DIR, 'results_scaled')

os.makedirs(WEIGHTS_DIR, exist_ok=True)
os.makedirs(os.path.join(WEIGHTS_DIR, 'checkpoints'), exist_ok=True)
os.makedirs(os.path.join(WEIGHTS_DIR, 'best'), exist_ok=True)  # Fix 7: best-diag checkpoint dir
os.makedirs(RESULTS_DIR, exist_ok=True)
sys.path.insert(0, SRC_DIR)

print('Environment :', 'Kaggle' if ON_KAGGLE else 'Local')
print('SRC_DIR     :', SRC_DIR)
print('SIM_LOG     :', SIM_LOG_PATH)
print('DATASET     :', DATASET_PATH)
print('WEIGHTS_DIR :', WEIGHTS_DIR)
print('RESULTS_DIR :', RESULTS_DIR)

In [ ]:
import time
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from collections import deque
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.patches import Patch

# Fix 9: Import from uav_env_3 (double-crash fix, W_UNKNOWN_FLOOR=0.1,
#         proximity tie-breaking, PBRS_SCALE=2.0) and networks_3
from uav_env_3 import (UAVFieldEnv, N_UAVS, N_SECTORS, GRID_ROWS, GRID_COLS,
                        E_MAX, TAU_DIAG, DAILY_STEPS_MAX, SURVIVAL_RATIO_THRESHOLD)
from networks_3 import SectorAttentionActor, CriticNetwork

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device :', DEVICE)
if torch.cuda.is_available():
    print('GPU    :', torch.cuda.get_device_name(0))
    print('VRAM   :', f'{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Hyperparameters

In [ ]:
N_EPISODES        = 400
K_EPOCHS          = 4             # Item 67: 6 → 4 (larger batches need fewer passes)
MINI_BATCH_SIZE   = 128           # Item 66: 24 → 128 (rollout is 2,880 steps)
GAMMA             = 0.99
GAE_LAMBDA        = 0.95
EPS_CLIP          = 0.2
ENTROPY_COEFF     = 0.001         # Item 89: 0.05 → 0.001 (fixes negative actor loss)
LR_ACTOR          = 3e-4
LR_CRITIC         = 3e-4
LR_MIN            = 1e-5
SAVE_INTERVAL     = 1_000
LOG_INTERVAL      = 50

# Item 84a/b: Split discovery bonus by diagnosis result
INFECTED_FOUND_BONUS = 30.0       # UAV diagnoses an infected sector
HEALTHY_FOUND_BONUS  = 5.0        # UAV diagnoses a healthy sector

# Item 85: HOVER_BONUS removed (was 5.0) — eliminates corner exploitation
OVERHOVER_PENALTY    = 5.0        # per step hovering on already-diagnosed sector

# Items 59–64: Daily sortie constants (must match uav_env_3.py)
CRASH_PENALTY        = 100.0      # Item 60
# Fix 5: SAFE_RETURN_BONUS 10.0 → 1.0
# Returning safely is now a lack of punishment (+1), not a goal.
# At 10.0, max achievable = 4 × 72 × 10 = 2,880 by never exploring → policy collapse.
# At 1.0, max = 288, well below one INFECTED_FOUND_BONUS (30) per discovery.
SAFE_RETURN_BONUS    = 1.0        # Fix 5: 10.0 → 1.0

# Item 65: PSI_NAV removed — superseded by PBRS in uav_env_3.py

UAV_COLORS = ['dodgerblue', 'darkorange', 'mediumseagreen', 'crimson']

## Environment & Model Initialisation

In [ ]:
# No reward monkey-patch needed — PBRS reward is built into uav_env_3.py
env = UAVFieldEnv(SIM_LOG_PATH, GRID_CFG_PATH, dataset_dir=DATASET_PATH)

actors = [SectorAttentionActor().to(DEVICE) for _ in range(N_UAVS)]
critic = CriticNetwork().to(DEVICE)

# FIX 6 (retained): Lower initial action log-std so actors start with std ~ 0.37
for actor in actors:
    nn.init.constant_(actor.action_log_std, -1.0)

actor_opts   = [torch.optim.Adam(a.parameters(), lr=LR_ACTOR) for a in actors]
critic_opt   = torch.optim.Adam(critic.parameters(), lr=LR_CRITIC)

actor_scheds = [
    torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=N_EPISODES, eta_min=LR_MIN)
    for opt in actor_opts
]
critic_sched = torch.optim.lr_scheduler.CosineAnnealingLR(
    critic_opt, T_max=N_EPISODES, eta_min=LR_MIN
)

actor_params  = sum(sum(p.numel() for p in a.parameters()) for a in actors)
critic_params = sum(p.numel() for p in critic.parameters())
print(f'Actor params  : {actor_params:,}  (x{N_UAVS} UAVs)')
print(f'Critic params : {critic_params:,}')
print(f'Total params  : {actor_params + critic_params:,}')
print(f'Episode length: {env.T} days x {DAILY_STEPS_MAX} steps/day = {env.total_steps} total steps')
print('Reward fn     : PBRS (explore/return) built into uav_env_3.py  (PBRS_SCALE=2.0)')
print('Actor log_std : -1.0 → std ~ 0.37')
print(f'SAFE_RETURN_BONUS : {SAFE_RETURN_BONUS}  (Fix 5: was 10.0)')

## Training Components

In [ ]:
class RunningMeanStd:
    """
    FIX 3 (retained): Online Welford mean/std for reward normalisation across episodes.
    """
    def __init__(self, epsilon: float = 1e-4):
        self.mean  = 0.0
        self.var   = 1.0
        self.count = epsilon

    def update(self, xs):
        batch = np.array(xs, dtype=np.float64).ravel()
        n     = batch.size
        if n == 0:
            return
        m     = batch.mean()
        v     = batch.var() if n > 1 else 0.0
        delta      = m - self.mean
        total_n    = self.count + n
        self.mean  = self.mean + delta * n / total_n
        self.var   = (self.var * self.count + v * n
                      + delta ** 2 * self.count * n / total_n) / total_n
        self.count = total_n

    def normalize(self, xs):
        return (np.array(xs, dtype=np.float32) - self.mean) / (np.sqrt(self.var) + 1e-8)

In [ ]:
class RolloutBuffer:
    def __init__(self):
        self.clear()

    def clear(self):
        self.obs       = [[] for _ in range(N_UAVS)]
        self.actions   = [[] for _ in range(N_UAVS)]
        self.log_probs = [[] for _ in range(N_UAVS)]
        self.rewards   = [[] for _ in range(N_UAVS)]
        self.values    = []
        self.dones     = []

    def store(self, obs_list, actions_list, log_probs_list, rewards_list, value, done):
        for u in range(N_UAVS):
            self.obs[u].append(obs_list[u])
            self.actions[u].append(actions_list[u])
            self.log_probs[u].append(log_probs_list[u])
            self.rewards[u].append(rewards_list[u])
        self.values.append(value)
        self.dones.append(done)

    def get_tensors(self):
        obs_t = [torch.FloatTensor(np.array(self.obs[u])).to(DEVICE)
                 for u in range(N_UAVS)]

        # FIX 1 (retained): FloatTensor for continuous actions
        acts_t = [torch.FloatTensor(np.array(self.actions[u])).to(DEVICE)
                  for u in range(N_UAVS)]

        # FIX 2 (retained): squeeze to prevent (batch,batch) broadcast
        lps_t = [torch.stack(self.log_probs[u]).squeeze(-1).to(DEVICE)
                 for u in range(N_UAVS)]

        rews_t  = [torch.FloatTensor(self.rewards[u]).to(DEVICE)
                   for u in range(N_UAVS)]
        vals_t  = torch.stack(self.values).view(-1).to(DEVICE)
        dones_t = torch.FloatTensor(self.dones).to(DEVICE)
        return obs_t, acts_t, lps_t, rews_t, vals_t, dones_t

In [ ]:
def compute_gae(rewards_list, values, dones, last_value):
    T_ep       = len(values)
    advantages = torch.zeros(T_ep).to(DEVICE)
    gae        = 0.0
    mean_rews  = torch.stack(rewards_list).mean(dim=0)
    values_ext = torch.cat([values.view(-1), last_value.view(1)])

    for t in reversed(range(T_ep)):
        mask          = 1.0 - dones[t]
        delta         = mean_rews[t] + GAMMA * values_ext[t + 1] * mask - values_ext[t]
        gae           = delta + GAMMA * GAE_LAMBDA * mask * gae
        advantages[t] = gae

    returns    = advantages + values
    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
    return advantages.detach(), returns.detach()

In [ ]:
def ppo_update(env, actors, critic, actor_opts, critic_opt, buffer):
    obs_t, acts_t, old_lps_t, rews_t, vals_t, dones_t = buffer.get_tensors()

    with torch.no_grad():
        last_joint = torch.cat(
            [torch.FloatTensor(env._get_obs(u)).to(DEVICE) for u in range(N_UAVS)]
        ).unsqueeze(0)
        last_value = critic(last_joint).squeeze()

    advantages, returns = compute_gae(rews_t, vals_t, dones_t, last_value)

    T_ep          = vals_t.shape[0]
    actor_losses  = []
    critic_losses = []

    for _ in range(K_EPOCHS):
        perm = torch.randperm(T_ep, device=DEVICE)
        for start in range(0, T_ep, MINI_BATCH_SIZE):
            mb_idx     = perm[start: start + MINI_BATCH_SIZE]
            mb_adv     = advantages[mb_idx]
            mb_returns = returns[mb_idx]

            for u in range(N_UAVS):
                mb_obs     = obs_t[u][mb_idx]
                mb_acts    = acts_t[u][mb_idx]
                mb_old_lps = old_lps_t[u][mb_idx].detach()

                new_lps, entropy = actors[u].get_log_prob_entropy(mb_obs, mb_acts)
                ratio      = torch.exp(new_lps - mb_old_lps)
                surr1      = ratio * mb_adv
                surr2      = torch.clamp(ratio, 1 - EPS_CLIP, 1 + EPS_CLIP) * mb_adv
                actor_loss = (-torch.min(surr1, surr2).mean()
                              - ENTROPY_COEFF * entropy)

                actor_opts[u].zero_grad()
                actor_loss.backward()
                nn.utils.clip_grad_norm_(actors[u].parameters(), 0.5)
                actor_opts[u].step()
                actor_losses.append(actor_loss.item())

            mb_joint    = torch.cat([obs_t[u][mb_idx] for u in range(N_UAVS)], dim=1)
            values_pred = critic(mb_joint).squeeze()
            critic_loss = nn.MSELoss()(values_pred, mb_returns)

            critic_opt.zero_grad()
            critic_loss.backward()
            nn.utils.clip_grad_norm_(critic.parameters(), 0.5)
            critic_opt.step()
            critic_losses.append(critic_loss.item())

    # Fix 8: returns mean over ALL K_EPOCHS × mini-batches (not just last batch)
    return np.mean(actor_losses), np.mean(critic_losses)

## Training Loop

In [ ]:
buffer         = RolloutBuffer()
reward_rms     = RunningMeanStd()
ep_rewards     = []
ep_discovered  = []
ep_crashes     = []          # Item 78: crashes per episode
ep_safe_returns = []         # Item 79: safe returns per episode
a_loss_log     = []
c_loss_log     = []
recent_rewards = deque(maxlen=50)
train_start    = time.time()

# Fix 7: best-diagnosis checkpoint tracker
best_diag_count = 0
best_dir        = os.path.join(WEIGHTS_DIR, 'best')

pbar = tqdm(range(1, N_EPISODES + 1), desc='Training', unit='ep')

for episode in pbar:
    obs       = env.reset()
    buffer.clear()
    ep_reward = 0.0
    n_crashes = 0              # Item 78
    n_safe_returns = 0         # Item 79

    # Item 71: inner loop over total steps (T days x DAILY_STEPS_MAX steps/day)
    for t in range(env.total_steps):
        joint_obs = torch.cat(
            [torch.FloatTensor(obs[u]).to(DEVICE) for u in range(N_UAVS)]
        ).unsqueeze(0)

        with torch.no_grad():
            value = critic(joint_obs)

        # Item 77: Removed energy <= 0 zero-velocity override.
        # Crash mechanic in uav_env_3.py handles energy depletion.
        actions, log_probs = [], []
        for u in range(N_UAVS):
            obs_t_u = torch.FloatTensor(obs[u]).to(DEVICE)
            with torch.no_grad():
                action, lp = actors[u].get_action(obs_t_u)
            actions.append(action)
            log_probs.append(lp)

        unknown_before = set(np.where(env.uav_status == 2)[0])
        next_obs, rewards, done, info = env.step(actions)

        # ── Item 84c: Split discovery bonus by diagnosis result ──
        # Only the diagnosing UAV gets the bonus.
        # Item 73: Disabled in return mode (prevents suicidal one-last-bonus).
        newly_found = unknown_before - set(np.where(env.uav_status == 2)[0])
        for sid in newly_found:
            r_sid, c_sid = env.sector_pos[sid]
            for u in range(N_UAVS):
                # Skip if UAV is in return mode
                if info['survival_ratio'][u] <= SURVIVAL_RATIO_THRESHOLD:
                    continue
                r_cont, c_cont = env.uav_pos[u]
                r_int = int(np.clip(round(r_cont), 0, GRID_ROWS - 1))
                c_int = int(np.clip(round(c_cont), 0, GRID_COLS - 1))
                if (r_int, c_int) == (r_sid, c_sid) and env.dwell[u] >= TAU_DIAG:
                    # Item 84c: Check new status for split bonus
                    if env.uav_status[sid] == 1:
                        rewards[u] += INFECTED_FOUND_BONUS
                    elif env.uav_status[sid] == 0:
                        rewards[u] += HEALTHY_FOUND_BONUS

        # Item 85b: HOVER_BONUS logic entirely removed (was corner exploitation).

        # Item 74: OVERHOVER_PENALTY — disabled in return mode.
        for u in range(N_UAVS):
            if info['survival_ratio'][u] <= SURVIVAL_RATIO_THRESHOLD:
                continue
            r_cont, c_cont = env.uav_pos[u]
            r_int = int(np.clip(round(r_cont), 0, GRID_ROWS - 1))
            c_int = int(np.clip(round(c_cont), 0, GRID_COLS - 1))
            current_sid = env.pos_to_sid[(r_int, c_int)]
            if env.dwell[u] > TAU_DIAG and env.uav_status[current_sid] != 2:
                rewards[u] -= OVERHOVER_PENALTY

        # Item 75: CRASH_PENALTY — one-shot on the step the UAV first crashes.
        # Fix 1 (env): env no longer applies this penalty internally, so no double-count.
        for u in range(N_UAVS):
            if info['newly_crashed'][u]:
                rewards[u] -= CRASH_PENALTY
                n_crashes += 1

        # Item 76: SAFE_RETURN_BONUS — once per UAV per day at daily reset.
        # Fix 5: bonus is 1.0 (was 10.0) — returning is lack of punishment, not a goal.
        for u in range(N_UAVS):
            if info['safely_returned'][u]:
                rewards[u] += SAFE_RETURN_BONUS
                n_safe_returns += 1

        buffer.store(obs, actions, log_probs, rewards, value, float(done))
        ep_reward += sum(rewards)
        obs        = next_obs
        if done:
            break

    # FIX 3 (retained): Running normalisation
    all_ep_rews = np.concatenate([np.array(buffer.rewards[u]) for u in range(N_UAVS)])
    reward_rms.update(all_ep_rews)
    for u in range(N_UAVS):
        buffer.rewards[u] = list(reward_rms.normalize(buffer.rewards[u]))

    a_loss, c_loss = ppo_update(env, actors, critic, actor_opts, critic_opt, buffer)

    for sched in actor_scheds:
        sched.step()
    critic_sched.step()

    # FIX 8 (retained): Track diagnosed + infected separately
    n_diagnosed = int((env.uav_status != 2).sum())
    n_inf_known = int((env.uav_status == 1).sum())
    # Item 80: avg diagnoses per day
    avg_diag_day = n_diagnosed / max(env.current_day, 1)

    ep_rewards.append(ep_reward)
    ep_discovered.append(n_diagnosed)
    ep_crashes.append(n_crashes)                # Item 78
    ep_safe_returns.append(n_safe_returns)      # Item 79
    a_loss_log.append(a_loss)
    c_loss_log.append(c_loss)
    recent_rewards.append(ep_reward)

    # Fix 7: Save best-diagnosis checkpoint whenever n_diagnosed improves
    if n_diagnosed > best_diag_count:
        best_diag_count = n_diagnosed
        for u in range(N_UAVS):
            torch.save(actors[u].state_dict(),
                       os.path.join(best_dir, f'actor{u}_best.pth'))
        torch.save(critic.state_dict(),
                   os.path.join(best_dir, 'critic_best.pth'))
        tqdm.write(f'  [best] episode {episode}  diag={n_diagnosed}/{N_SECTORS}  (new record)')

    avg = np.mean(recent_rewards)
    # Item 81: Updated postfix with new metrics
    pbar.set_postfix(ordered_dict={
        'reward':  f'{ep_reward:.1f}',
        'avg50':   f'{avg:.1f}',
        'diag':    f'{n_diagnosed}/{N_SECTORS}',
        'inf':     n_inf_known,
        'crash':   n_crashes,
        'safe':    n_safe_returns,
        'a_loss':  f'{a_loss:.4f}',
        'c_loss':  f'{c_loss:.4f}',
    })

    if episode % LOG_INTERVAL == 0 or episode == 1:
        elapsed = time.time() - train_start
        eta_sec = (elapsed / episode) * (N_EPISODES - episode)
        lr_now  = actor_opts[0].param_groups[0]['lr']
        # Item 81: Updated log line with crashes, safe returns, avg_diag/day
        tqdm.write(
            f'  Ep {episode:>6}/{N_EPISODES}'
            f'  reward={ep_reward:>9.2f}'
            f'  avg50={avg:>9.2f}'
            f'  diag={n_diagnosed:>3}/{N_SECTORS}'
            f'  inf={n_inf_known:>3}'
            f'  crash={n_crashes}'
            f'  safe={n_safe_returns}'
            f'  diag/day={avg_diag_day:.1f}'
            f'  a_loss={a_loss:.4f}'
            f'  c_loss={c_loss:.4f}'
            f'  lr={lr_now:.2e}'
            f'  ETA={eta_sec/60:.1f}min'
        )

    if episode % SAVE_INTERVAL == 0:
        ckpt_dir = os.path.join(WEIGHTS_DIR, 'checkpoints')
        for u in range(N_UAVS):
            torch.save(actors[u].state_dict(),
                       os.path.join(ckpt_dir, f'actor{u}_ep{episode}.pth'))
        torch.save(critic.state_dict(),
                   os.path.join(ckpt_dir, f'critic_ep{episode}.pth'))
        tqdm.write(f'  [checkpoint] episode {episode}')

total_time = time.time() - train_start
print(f'\nTraining complete in {total_time/60:.1f} min')
print(f'Best reward           : {max(ep_rewards):.2f}')
print(f'Final avg(50)         : {np.mean(list(recent_rewards)):.2f}')
print(f'Best total diagnosed  : {best_diag_count} / {N_SECTORS}  (Fix 7: best checkpoint saved)')
print(f'Total crashes         : {sum(ep_crashes)}')
print(f'Total safe returns    : {sum(ep_safe_returns)}')

## Training Results

In [ ]:
def moving_avg(data, w=50):
    return np.convolve(data, np.ones(w) / w, mode='valid')

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('MAPPO Training (Scaled — Daily Sortie) — 10x10 Grid, 4 UAVs', fontsize=13)

axes[0, 0].plot(ep_rewards, alpha=0.2, color='steelblue')
if len(ep_rewards) >= 50:
    axes[0, 0].plot(moving_avg(ep_rewards), color='steelblue', label='MA(50)')
axes[0, 0].set_title('Total Reward per Episode')
axes[0, 0].set_xlabel('Episode')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(ep_discovered, alpha=0.2, color='tomato')
if len(ep_discovered) >= 50:
    axes[0, 1].plot(moving_avg(ep_discovered), color='tomato', label='MA(50)')
axes[0, 1].set_title('Total Sectors Diagnosed per Episode')
axes[0, 1].set_xlabel('Episode')
axes[0, 1].set_ylabel('Sectors (healthy + infected)')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

if len(a_loss_log) >= 50:
    axes[1, 0].plot(moving_avg(a_loss_log), color='darkorange')
axes[1, 0].set_title('Actor Loss (MA-50)')
axes[1, 0].set_xlabel('Episode')
axes[1, 0].grid(True, alpha=0.3)

if len(c_loss_log) >= 50:
    axes[1, 1].plot(moving_avg(c_loss_log), color='mediumseagreen')
axes[1, 1].set_title('Critic Loss (MA-50)')
axes[1, 1].set_xlabel('Episode')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
out_path = os.path.join(RESULTS_DIR, 'training_curves.png')
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'[saved] {out_path}')

## Save Weights

In [ ]:
# Save final weights
for u in range(N_UAVS):
    path = os.path.join(WEIGHTS_DIR, f'actor{u}_final.pth')
    torch.save(actors[u].state_dict(), path)
    print(f'[saved] {path}')

path = os.path.join(WEIGHTS_DIR, 'critic_final.pth')
torch.save(critic.state_dict(), path)
print(f'[saved] {path}')

# Fix 7: best-diag weights saved live during training to WEIGHTS_DIR/best/
print(f'\n[best] Best-diagnosis weights at: {best_dir}')
print(f'[best] Peak diagnosed sectors   : {best_diag_count} / {N_SECTORS}')

## Evaluation

**Fix 6:** Uses `dist.loc` (deterministic mean action) instead of `get_action()` (stochastic sample).
Since `SectorAttentionActor.forward()` returns a `torch.distributions.Normal` object,
`dist.loc` accesses `mu` directly — no random noise during evaluation.

**Fix 7:** Loads from best-diagnosis checkpoint (`best/actor*_best.pth`) rather than final weights.

In [ ]:
eval_actors = [SectorAttentionActor().to(DEVICE) for _ in range(N_UAVS)]
for u in range(N_UAVS):
    # Fix 7: Load best-diagnosis checkpoint, not final
    path = os.path.join(best_dir, f'actor{u}_best.pth')
    eval_actors[u].load_state_dict(torch.load(path, map_location=DEVICE))
    eval_actors[u].eval()
    print(f'Loaded (best): {path}')

eval_env     = UAVFieldEnv(SIM_LOG_PATH, GRID_CFG_PATH, dataset_dir=DATASET_PATH)
obs          = eval_env.reset()
eval_history = []
total_reward = 0.0
eval_crashes = 0         # Item 82
eval_safe_returns = 0    # Item 82

for t in range(eval_env.total_steps):
    actions = []
    for u in range(N_UAVS):
        obs_t = torch.FloatTensor(obs[u]).to(DEVICE)
        with torch.no_grad():
            # Fix 6: Deterministic evaluation — use distribution mean (dist.loc)
            # forward() returns Normal(mu, std); dist.loc == mu (no sampling noise)
            dist   = eval_actors[u].forward(obs_t.unsqueeze(0))
            action = dist.loc.squeeze().cpu().numpy()
        actions.append(action)

    obs, rewards, done, info = eval_env.step(actions)
    total_reward += sum(rewards)

    # Item 82: track crashes and safe returns during eval
    eval_crashes      += sum(info['newly_crashed'])
    eval_safe_returns += sum(info['safely_returned'])

    eval_history.append({
        't':            t + 1,
        'current_day':  info['current_day'],
        'daily_step':   info['daily_step'],
        'uav_pos':      list(eval_env.uav_pos),
        'uav_status':   eval_env.uav_status.copy(),
        'true_status':  eval_env.true_status.copy(),
        'risk_weights': eval_env.w.copy(),
        'energy':       list(eval_env.energy),
        'reward':       sum(rewards),
    })
    if done:
        break

n_vis = int((eval_env.uav_status != 2).sum())
n_inf = int((eval_env.uav_status == 1).sum())
n_tru = int((eval_env.true_status == 1).sum())
print(f'\nEvaluation complete')
print(f'  Total reward    : {total_reward:.2f}')
print(f'  Sectors visited : {n_vis} / {N_SECTORS}')
print(f'  Infected found  : {n_inf}')
print(f'  True infected   : {n_tru}')
print(f'  Detection rate  : {n_inf / max(n_tru, 1) * 100:.1f}%')
print(f'  Crashes         : {eval_crashes}')

In [ ]:
timesteps     = [s['t'] for s in eval_history]
n_visited     = [(s['uav_status'] != 2).sum() for s in eval_history]
n_inf_found   = [(s['uav_status'] == 1).sum() for s in eval_history]
true_infected = [(s['true_status'] == 1).sum() for s in eval_history]
mean_risk     = [s['risk_weights'].mean() for s in eval_history]
rewards_step  = [s['reward'] for s in eval_history]
cum_reward    = np.cumsum(rewards_step)
energy        = [[s['energy'][u] for s in eval_history] for u in range(N_UAVS)]

fig, axes = plt.subplots(4, 2, figsize=(18, 22))
fig.suptitle('MAPPO Results (Scaled — Daily Sortie) — 10x10 Grid, 4 UAVs, 72 Days',
             fontsize=15, fontweight='bold')

# Panel 1: Coverage
ax = axes[0, 0]
ax.plot(timesteps, n_visited, color='steelblue', linewidth=2)
ax.axhline(N_SECTORS, color='gray', linestyle='--', alpha=0.7, label=f'Total ({N_SECTORS})')
ax.fill_between(timesteps, n_visited, alpha=0.15, color='steelblue')
ax.set_title('Field Coverage Over Time')
ax.set_xlabel('Step')
ax.set_ylabel('Sectors Visited')
ax.legend()
ax.grid(True, alpha=0.3)

# Panel 2: Detection
ax = axes[0, 1]
ax.plot(timesteps, true_infected, color='tomato', linestyle='--', linewidth=2, label='True Infected')
ax.plot(timesteps, n_inf_found, color='darkred', linewidth=2, label='Found by UAVs')
ax.fill_between(timesteps, n_inf_found, true_infected, alpha=0.2, color='tomato', label='Detection Gap')
ax.set_title('Infected Found vs Ground Truth')
ax.set_xlabel('Step')
ax.legend()
ax.grid(True, alpha=0.3)

# Panel 3: Battery (sawtooth pattern with daily recharge)
ax = axes[1, 0]
for u in range(N_UAVS):
    ax.plot(timesteps, energy[u], color=UAV_COLORS[u], linewidth=1.0, alpha=0.7, label=f'UAV {u}')
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_title('Battery per UAV (daily recharge)')
ax.set_xlabel('Step')
ax.set_ylabel('Energy Units')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Panel 4: Mean Risk
ax = axes[1, 1]
ax.plot(timesteps, mean_risk, color='darkorchid', linewidth=2)
ax.fill_between(timesteps, mean_risk, alpha=0.15, color='darkorchid')
ax.set_title('Mean Field Risk Weight Over Time')
ax.set_xlabel('Step')
ax.set_ylabel('Mean w[k]')
ax.grid(True, alpha=0.3)

# Panel 5: Cumulative Reward
ax = axes[2, 0]
ax.plot(timesteps, cum_reward, color='teal', linewidth=2)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.fill_between(timesteps, cum_reward, 0,
                where=np.array(cum_reward) >= 0, alpha=0.2, color='teal', label='Positive')
ax.fill_between(timesteps, cum_reward, 0,
                where=np.array(cum_reward) < 0, alpha=0.2, color='red', label='Negative')
ax.set_title('Cumulative Reward')
ax.set_xlabel('Step')
ax.legend()
ax.grid(True, alpha=0.3)

# Panel 6: Per-Step Reward
ax = axes[2, 1]
colors_r = ['mediumseagreen' if r >= 0 else 'tomato' for r in rewards_step]
ax.bar(timesteps, rewards_step, color=colors_r, alpha=0.8, width=0.9)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('Per-Step Reward')
ax.set_xlabel('Step')
ax.grid(True, alpha=0.2, axis='y')

# Panel 7: Trajectory Map
ax = axes[3, 0]
ax.set_xlim(-0.5, GRID_COLS - 0.5)
ax.set_ylim(-0.5, GRID_ROWS - 0.5)
ax.set_aspect('equal')
ax.invert_yaxis()
ax.set_xticks(range(GRID_COLS))
ax.set_yticks(range(GRID_ROWS))
ax.grid(True, alpha=0.25)
ax.set_title('UAV Trajectories (o=start, s=end)')
ax.set_xlabel('Column')
ax.set_ylabel('Row')

final_status = eval_history[-1]['uav_status']
for sid in range(N_SECTORS):
    r, c = sid // GRID_COLS, sid % GRID_COLS
    color = ('#ffe0e0' if final_status[sid] == 1 else
             '#e0ffe0' if final_status[sid] == 0 else '#f5f5f5')
    rect = plt.Rectangle((c - 0.48, r - 0.48), 0.96, 0.96,
                          facecolor=color, edgecolor='#cccccc', linewidth=0.5)
    ax.add_patch(rect)

init_true = eval_history[0]['true_status']
for sid in range(N_SECTORS):
    if init_true[sid] == 1:
        r, c = sid // GRID_COLS, sid % GRID_COLS
        ax.text(c, r, '*', ha='center', va='center', fontsize=6, color='darkred', alpha=0.6)

for u in range(N_UAVS):
    pr = [s['uav_pos'][u][0] for s in eval_history]
    pc = [s['uav_pos'][u][1] for s in eval_history]
    ax.plot(pc, pr, color=UAV_COLORS[u], linewidth=1.2, alpha=0.55, label=f'UAV {u}')
    ax.plot(pc[0],  pr[0],  'o', color=UAV_COLORS[u], markersize=8, zorder=5)
    ax.plot(pc[-1], pr[-1], 's', color=UAV_COLORS[u], markersize=8, zorder=5)

legend_patches = [
    Patch(facecolor='#ffe0e0', label='Infected (found)'),
    Patch(facecolor='#e0ffe0', label='Healthy (found)'),
    Patch(facecolor='#f5f5f5', label='Unknown'),
]
uav_lines = [plt.Line2D([0], [0], color=UAV_COLORS[u], linewidth=1.5, label=f'UAV {u}')
             for u in range(N_UAVS)]
ax.legend(handles=legend_patches + uav_lines, loc='lower right', fontsize=7, ncol=2)

# Panel 8: Final Risk Heatmap
ax = axes[3, 1]
ax.set_xlim(-0.5, GRID_COLS - 0.5)
ax.set_ylim(-0.5, GRID_ROWS - 0.5)
ax.set_aspect('equal')
ax.invert_yaxis()
ax.set_xticks(range(GRID_COLS))
ax.set_yticks(range(GRID_ROWS))
ax.set_title('Final Risk Weight Heatmap')
ax.set_xlabel('Column')
ax.set_ylabel('Row')

final_risk = eval_history[-1]['risk_weights']
norm       = Normalize(vmin=0, vmax=1)
cmap       = plt.cm.RdYlGn_r

for sid in range(N_SECTORS):
    r, c  = sid // GRID_COLS, sid % GRID_COLS
    color = cmap(norm(final_risk[sid]))
    rect  = plt.Rectangle((c - 0.48, r - 0.48), 0.96, 0.96,
                           facecolor=color, edgecolor='white', linewidth=0.5)
    ax.add_patch(rect)

for u, (r, c) in enumerate(eval_history[-1]['uav_pos']):
    ax.text(c, r, str(u), ha='center', va='center', fontsize=9, fontweight='bold',
            color='white',
            bbox=dict(boxstyle='circle,pad=0.1', facecolor=UAV_COLORS[u], edgecolor='none'))

sm = ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
plt.colorbar(sm, ax=ax, label='Risk Weight', fraction=0.025, pad=0.03)

plt.tight_layout(rect=[0, 0, 1, 0.97])
out_path = os.path.join(RESULTS_DIR, 'results_graph.png')
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'[saved] {out_path}')

In [ ]:
print('=' * 60)
print('  FINAL EVALUATION SUMMARY')
print('=' * 60)
print(f'  Grid            : {GRID_ROWS}x{GRID_COLS} = {N_SECTORS} sectors')
print(f'  UAVs            : {N_UAVS}')
print(f'  Episode length  : {eval_env.T} days x {DAILY_STEPS_MAX} steps/day')
print(f'  Weights from    : best-diagnosis checkpoint ({best_diag_count}/{N_SECTORS} sectors)')
print(f'  Total reward    : {total_reward:.2f}')
print(f'  Sectors visited : {n_vis} / {N_SECTORS}  ({n_vis/N_SECTORS*100:.1f}%)')
print(f'  Infected found  : {n_inf}')
print(f'  True infected   : {n_tru}')
print(f'  Detection rate  : {n_inf / max(n_tru, 1) * 100:.1f}%')
# Item 82/83: crashes and safe returns replace per-UAV energy display
print(f'  Crashes         : {eval_crashes}')
print(f'  Safe returns    : {eval_safe_returns}')
print('=' * 60)